In [ ]:
!git clone --depth 1 https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile realesrgan.py enhance/*.py
!cd /kaggle/working/Real-ESRGAN && pytest -q

In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime2/cm_4.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
QUALITY_PRESET = "safe"       # baseline / safe
SCALE = 2
FPS = "source"

START_TIME = 3 * 60 + 15
TEST_SECONDS = 10
RUN_BOTH_10S_TESTS = True
PROGRESS_INTERVAL = 60.0

NATIVE_ANALYSIS = "off"       # off / report / auto
NATIVE_SAMPLES = 5
NATIVE_MIN_HEIGHT = 500
NATIVE_MAX_HEIGHT = 1080
NATIVE_KERNELS = "bilinear,bicubic,lanczos"
NATIVE_CONFIDENCE = 0.85
NATIVE_HEIGHT = 0
NATIVE_KERNEL = "auto"
DESCALE = False

INPUT_WIDTH = 0
INPUT_HEIGHT = 0
TILE_SIZE = 256                # 0=整帧；T4 建议先用 256
TILE_PAD = 10
PRE_PAD = 0
TILE_VERIFY_COVERAGE = True
BATCH_SIZE = 4
GPU_IDS = "0,1"
TTA_BATCH_SIZE = 1

BACK_PROJECTION_STRENGTH = 0.2
BACK_PROJECTION_KERNEL = "lanczos"
BACK_PROJECTION_CLAMP = 0.05
DEHALO_RADIUS = 2
RANGE_RADIUS = 2
OVERSHOOT = 1.0
UNDERSHOOT = 1.0

COLOR_POLICY = "preserve"     # preserve / bt709
HDR_POLICY = "reject"         # reject / passthrough

ANIME4K = False
ANIME4K_SHADER_DIR = ""
ANIME4K_SHADERS = ""
ANIME4K_STRENGTH = 1.0

VIDEO_CODEC = "hevc_nvenc"
OUTPUT_PIX_FMT = "auto"
CRF = 18
PRESET = "medium"
CQ = 18
NVENC_PRESET = "p7"
ENCODE_GPU = 0
AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"

# Advanced CLI overrides. These take precedence over the selected preset.
EXTRA_ARGS = []


In [ ]:
import shlex
import subprocess
import sys
from pathlib import Path

presets = ["baseline", "safe"] if RUN_BOTH_10S_TESTS and TEST_SECONDS == 10 else [QUALITY_PRESET]
for quality_preset in presets:
    output = Path(OUTPUT_VIDEO)
    if len(presets) > 1:
        output = output.with_name(f"{output.stem}_{quality_preset}{output.suffix}")
    command = [
        sys.executable, "/kaggle/working/Real-ESRGAN/realesrgan.py",
        "--input", INPUT_VIDEO, "--output", str(output),
        "--model", MODEL, "--model-path", MODEL_PATH,
        "--quality-preset", quality_preset, "--scale", str(SCALE), "--fps", FPS,
        "--fp16", "--channels-last",
        "--native-analysis", NATIVE_ANALYSIS, "--native-samples", str(NATIVE_SAMPLES),
        "--native-min-height", str(NATIVE_MIN_HEIGHT), "--native-max-height", str(NATIVE_MAX_HEIGHT),
        "--native-kernels", NATIVE_KERNELS, "--native-confidence", str(NATIVE_CONFIDENCE),
        "--native-height", str(NATIVE_HEIGHT), "--native-kernel", NATIVE_KERNEL,
        "--descale" if DESCALE else "--no-descale",
        "--input-width", str(INPUT_WIDTH), "--input-height", str(INPUT_HEIGHT),
        "--tile-size", str(TILE_SIZE), "--tile-pad", str(TILE_PAD), "--pre-pad", str(PRE_PAD),
        "--tile-verify-coverage" if TILE_VERIFY_COVERAGE else "--no-tile-verify-coverage",
        "--batch-size", str(BATCH_SIZE), "--gpu-ids", GPU_IDS,
        "--tta-batch-size", str(TTA_BATCH_SIZE),
        "--back-projection-strength", str(BACK_PROJECTION_STRENGTH),
        "--back-projection-kernel", BACK_PROJECTION_KERNEL,
        "--back-projection-clamp", str(BACK_PROJECTION_CLAMP),
        "--dehalo-radius", str(DEHALO_RADIUS), "--range-radius", str(RANGE_RADIUS),
        "--overshoot", str(OVERSHOOT), "--undershoot", str(UNDERSHOOT),
        "--color-policy", COLOR_POLICY, "--hdr-policy", HDR_POLICY,
        "--anime4k" if ANIME4K else "--no-anime4k",
        "--anime4k-shader-dir", ANIME4K_SHADER_DIR,
        "--anime4k-shaders", ANIME4K_SHADERS, "--anime4k-strength", str(ANIME4K_STRENGTH),
        "--video-codec", VIDEO_CODEC, "--output-pix-fmt", OUTPUT_PIX_FMT,
        "--crf", str(CRF), "--preset", PRESET, "--cq", str(CQ),
        "--nvenc-preset", NVENC_PRESET, "--encode-gpu", str(ENCODE_GPU),
        "--audio-codec", AUDIO_CODEC, "--audio-bitrate", AUDIO_BITRATE,
        "--start-time", str(START_TIME), "--test-seconds", str(TEST_SECONDS),
        "--progress-interval", str(PROGRESS_INTERVAL), "--ffmpeg-bin", "ffmpeg",
        "--ffprobe-bin", "ffprobe", *EXTRA_ARGS,
    ]
    print("[command]", shlex.join(command), flush=True)
    subprocess.run(command, check=True)


## Optional Descale/getnative installation
Run only when `DESCALE=True`. The cell checks Python compatibility first; no resize fallback is used if VapourSynth is unavailable.


In [ ]:
import sys
if sys.version_info < (3, 12):
    raise RuntimeError("The pinned optional VapourSynth packages require Python 3.12 or newer in this notebook.")
!pip install -q wrapt VapourSynth==77 vapoursynth-descale==12 vapoursynth-ffms2==5.2.1 getnative==3.3.0
!vapoursynth config
!vapoursynth check-env
!command -v vspipe
!getnative --help >/dev/null
